In [1]:
# -- %%sql
# -- delete from lh_vv_bronze.dbo.brz_watermark where watermark_id = 'visitvn_poi_pipeline_v2';

StatementMeta(, d49eae0b-4fc8-4c38-96bb-cfbbcae9e9b0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
# Fabric Notebook: NB_00_POI_PIPELINE_ORCHESTRATOR
# Default Lakehouse: lh_vv_silver (MANDATORY for NB_00 -> NB_60)
# Purpose:
#   - Manage deterministic incremental window for POI Pipeline V2
#   - Orchestrate NB_10 -> NB_60
#   - Commit watermark ONLY after all processing notebooks succeed
#   - Track watermark_sequence / last_run_id / error_message
#
# Business contract:
#   watermark_from < crawled_at <= watermark_to
#
# IMPORTANT:
#   - Do NOT use INSERT * / UPDATE SET * for brz_watermark.
#   - watermark_value and watermark_sequence advance only on successful full pipeline.
#   - A failed run keeps the previous watermark intact so the same batch can be retried.

from datetime import datetime, timezone
import uuid
import time
import json

# ============================================================
# CELL 1 - CONFIG
# ============================================================

WATERMARK_ID = "bronze_to_silver_gold_v2"
FLOW_NAME = "visitvn_poi_pipeline_v2"

BRONZE_RAW = "sgr_visitvn_dev.lh_vv_bronze.dbo.poi_raw_event"
CONTROL_WM = "sgr_visitvn_dev.lh_vv_bronze.dbo.brz_watermark"
CONTROL_RUN = "sgr_visitvn_dev.lh_vv_bronze.dbo.brz_pipeline_run"

# IMPORTANT:
# Attach NB_00, NB_10, NB_20, NB_30, NB_40, NB_50 and NB_60 to the SAME
# default Lakehouse: lh_vv_silver.
# Do not pass useRootDefaultLakehouse in notebook arguments. Keeping a single
# execution context avoids Fabric notebook-reference/default-Lakehouse conflicts.

# Child notebook names / paths.
# Update these values if notebooks are stored inside a Fabric folder.
NB_10 = "NB_10_BRONZE_TO_SILVER_PREPROCESS"
NB_20 = "NB_20_SILVER_POI_COMPONENTS"
NB_30 = "NB_30_SILVER_POI_ENTITY_ENRICHMENT"
NB_40 = "NB_40_GOLD_POI_SERVING"
NB_50 = "NB_50_GOLD_POI_AI"
NB_60 = "NB_60_POI_DATA_QUALITY"

NOTEBOOK_TIMEOUT_SECONDS = 7200

# Fabric blocks parent -> child execution when notebooks are attached to
# different default Lakehouses unless this option is enabled.
# All V2 notebooks use fully-qualified table names, so the root notebook's
# default Lakehouse is only an execution context, not a table-resolution dependency.

DEFAULT_WATERMARK = "2026-09-14 00:00:00"


def sql_escape(value):
    """Escape a Python value before embedding into Spark SQL string literals."""
    if value is None:
        return None
    return str(value).replace("'", "''")


def ts_literal(value):
    """Return Spark SQL TIMESTAMP('...') literal for datetime/string value."""
    return f"TIMESTAMP('{sql_escape(value)}')"




# ============================================================
# CELL 2 - BOOTSTRAP CONTROL TABLES
# ============================================================

# Safe no-op when the table already exists.
# Schema matches the current brz_watermark structure used by the project.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_WM} (
    watermark_id       STRING,
    table_name         STRING,
    watermark_column   STRING,
    watermark_value    TIMESTAMP,
    update_time        TIMESTAMP,
    status             STRING,
    flow_name          STRING,
    watermark_sequence BIGINT,
    last_run_id        STRING,
    error_message      STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_RUN} (
    run_id          STRING,
    flow_name       STRING,
    status          STRING,
    watermark_from  TIMESTAMP,
    watermark_to    TIMESTAMP,
    raw_event_count BIGINT,
    current_notebook STRING,
    started_at      TIMESTAMP,
    ended_at        TIMESTAMP,
    duration        STRING,
    error_message   STRING
) USING DELTA
""")

# Existing environments may already have brz_pipeline_run without `duration`.
# CREATE TABLE IF NOT EXISTS does not evolve an existing table schema, therefore
# add the column explicitly as a backward-compatible bootstrap step.
try:
    spark.sql(f"ALTER TABLE {CONTROL_RUN} ADD COLUMNS (duration STRING)")
except Exception:
    # Expected when the column already exists.
    pass

# Bootstrap the V2 watermark record.
# Explicit INSERT mapping is intentional: brz_watermark has metadata fields such as
# watermark_sequence, last_run_id and error_message.
spark.sql(f"""
MERGE INTO {CONTROL_WM} AS tgt
USING (
    SELECT
        '{sql_escape(WATERMARK_ID)}'       AS watermark_id,
        'poi_raw_event'                    AS table_name,
        'crawled_at'                       AS watermark_column,
        TIMESTAMP('{DEFAULT_WATERMARK}')   AS watermark_value,
        current_timestamp()                AS update_time,
        'active'                           AS status,
        '{sql_escape(FLOW_NAME)}'          AS flow_name,
        CAST(0 AS BIGINT)                  AS watermark_sequence,
        CAST(NULL AS STRING)               AS last_run_id,
        CAST(NULL AS STRING)               AS error_message
) AS src
ON tgt.watermark_id = src.watermark_id
WHEN NOT MATCHED THEN INSERT (
    watermark_id,
    table_name,
    watermark_column,
    watermark_value,
    update_time,
    status,
    flow_name,
    watermark_sequence,
    last_run_id,
    error_message
)
VALUES (
    src.watermark_id,
    src.table_name,
    src.watermark_column,
    src.watermark_value,
    src.update_time,
    src.status,
    src.flow_name,
    src.watermark_sequence,
    src.last_run_id,
    src.error_message
)
""")


# ============================================================
# CELL 3 - READ CURRENT WATERMARK
# ============================================================

# Do NOT filter by status='active'.
# A previous failed run leaves status='error', but the same watermark must still be
# readable so the batch can be retried.
wm_row = spark.sql(f"""
SELECT
    COALESCE(watermark_value, TIMESTAMP('{DEFAULT_WATERMARK}')) AS watermark_value,
    COALESCE(watermark_sequence, 0) AS watermark_sequence,
    status,
    last_run_id,
    error_message
FROM {CONTROL_WM}
WHERE watermark_id = '{sql_escape(WATERMARK_ID)}'
LIMIT 1
""").first()

if wm_row is None:
    raise RuntimeError(f"Watermark row not found after bootstrap: {WATERMARK_ID}")

watermark_from = wm_row["watermark_value"]
watermark_sequence_before = wm_row["watermark_sequence"]
previous_status = wm_row["status"]
previous_run_id = wm_row["last_run_id"]



# ============================================================
# CELL 4 - CAPTURE FIXED WATERMARK_TO
# ============================================================

# Capture watermark_to once at the beginning of the run.
# Events arriving after this point are intentionally left for the next run.
watermark_to_row = spark.sql(f"""
SELECT MAX(crawled_at) AS watermark_to
FROM {BRONZE_RAW}
WHERE crawled_at > {ts_literal(watermark_from)}
""").first()

watermark_to = watermark_to_row["watermark_to"] if watermark_to_row else None

if watermark_to is None:

    # Normalize a stale running/error status back to active because there is currently
    # no unprocessed data beyond the committed watermark.
    spark.sql(f"""
    UPDATE {CONTROL_WM}
    SET
        status        = 'active',
        error_message = NULL,
        update_time   = current_timestamp(),
        flow_name     = '{sql_escape(FLOW_NAME)}'
    WHERE watermark_id = '{sql_escape(WATERMARK_ID)}'
    """)

    notebookutils.notebook.exit("NO_NEW_DATA")

# Count exactly the fixed window that every child notebook will process.
raw_event_count = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {BRONZE_RAW}
WHERE crawled_at > {ts_literal(watermark_from)}
  AND crawled_at <= {ts_literal(watermark_to)}
""").first()["cnt"]

if raw_event_count == 0:
    # Defensive guard: MAX(crawled_at) was not NULL, therefore this should not happen.
    raise RuntimeError(
        f"Invalid fixed window: watermark_from={watermark_from}, "
        f"watermark_to={watermark_to}, raw_event_count=0"
    )



# ============================================================
# CELL 5 - CREATE RUN
# ============================================================

run_id = str(uuid.uuid4())

# Monotonic timer is used for duration calculation because it is not affected by
# wall-clock/NTP adjustments. `started_at`/`ended_at` in Delta remain wall-clock timestamps.
pipeline_perf_started = time.perf_counter()
notebook_metrics = {}

spark.sql(f"""
INSERT INTO {CONTROL_RUN} (
    run_id,
    flow_name,
    status,
    watermark_from,
    watermark_to,
    raw_event_count,
    current_notebook,
    started_at,
    ended_at,
    duration,
    error_message
)
VALUES (
    '{sql_escape(run_id)}',
    '{sql_escape(FLOW_NAME)}',
    'RUNNING',
    {ts_literal(watermark_from)},
    {ts_literal(watermark_to)},
    {int(raw_event_count)},
    NULL,
    current_timestamp(),
    NULL,
    NULL,
    NULL
)
""")

# Mark current control state as running WITHOUT advancing the committed watermark.
spark.sql(f"""
UPDATE {CONTROL_WM}
SET
    status        = 'running',
    last_run_id   = '{sql_escape(run_id)}',
    error_message = NULL,
    update_time   = current_timestamp(),
    flow_name     = '{sql_escape(FLOW_NAME)}'
WHERE watermark_id = '{sql_escape(WATERMARK_ID)}'
""")



# ============================================================
# CELL 6 - CHILD NOTEBOOK ARGUMENTS / DEPENDENCIES
# ============================================================

# All downstream notebooks MUST use these exact values.
# Contract:
#     watermark_from < crawled_at <= watermark_to
args = {
    "run_id": run_id,
    "watermark_from": str(watermark_from),
    "watermark_to": str(watermark_to),
}

steps = [
    NB_10,
    NB_20,
    NB_30,
    NB_40,
    NB_50,
    NB_60,
]

# ============================================================
# CELL 7 - ORCHESTRATE V2
# ============================================================

def build_duration_json():
    total_duration_ms = int(round(
        (time.perf_counter() - pipeline_perf_started) * 1000
    ))

    if notebook_metrics:
        slowest_notebook, slowest_info = max(
            notebook_metrics.items(),
            key=lambda kv: kv[1]["duration_ms"]
        )
        slowest_duration_ms = int(slowest_info["duration_ms"])
    else:
        slowest_notebook = None
        slowest_duration_ms = None

    return json.dumps(
        {
            "total_duration_ms": total_duration_ms,
            "slowest_notebook": slowest_notebook,
            "slowest_duration_ms": slowest_duration_ms,
            "notebooks": notebook_metrics
        },
        ensure_ascii=False,
        separators=(",", ":")
    )


watermark_committed = False

try:
    for nb in steps:
        # Keep this single control write per child for failure traceability.
        spark.sql(f"""
        UPDATE {CONTROL_RUN}
        SET current_notebook = '{sql_escape(nb)}'
        WHERE run_id = '{sql_escape(run_id)}'
        """)

        nb_started = time.perf_counter()

        try:
            notebookutils.notebook.run(
                nb,
                NOTEBOOK_TIMEOUT_SECONDS,
                args
            )

            notebook_metrics[nb] = {
                "duration_ms": int(round(
                    (time.perf_counter() - nb_started) * 1000
                )),
                "status": "SUCCESS"
            }

        except Exception:
            notebook_metrics[nb] = {
                "duration_ms": int(round(
                    (time.perf_counter() - nb_started) * 1000
                )),
                "status": "FAILED"
            }
            raise

    # Commit watermark only after every child succeeds.
    spark.sql(f"""
    UPDATE {CONTROL_WM}
    SET
        watermark_value    = {ts_literal(watermark_to)},
        watermark_sequence = COALESCE(watermark_sequence, 0) + 1,
        last_run_id        = '{sql_escape(run_id)}',
        error_message      = NULL,
        status             = 'active',
        update_time        = current_timestamp(),
        flow_name          = '{sql_escape(FLOW_NAME)}'
    WHERE watermark_id = '{sql_escape(WATERMARK_ID)}'
    """)

    watermark_committed = True

    # Persist metrics once on success.
    duration_json = build_duration_json()

    spark.sql(f"""
    UPDATE {CONTROL_RUN}
    SET
        status           = 'SUCCESS',
        current_notebook = NULL,
        ended_at         = current_timestamp(),
        duration         = '{sql_escape(duration_json)}',
        error_message    = NULL
    WHERE run_id = '{sql_escape(run_id)}'
    """)

    # Best-effort run-scoped cleanup.
    try:
        spark.sql(
            f"DELETE FROM _stage_raw_event "
            f"WHERE run_id = '{sql_escape(run_id)}'"
        )
        spark.sql(
            f"DELETE FROM _stage_poi_language "
            f"WHERE run_id = '{sql_escape(run_id)}'"
        )
    except Exception:
        pass

    notebookutils.notebook.exit(f"SUCCESS:{run_id}")

except Exception as e:
    err = sql_escape(str(e))[:4000]
    duration_json = build_duration_json()

    if not watermark_committed:
        try:
            spark.sql(f"""
            UPDATE {CONTROL_WM}
            SET
                status        = 'error',
                last_run_id   = '{sql_escape(run_id)}',
                error_message = '{err}',
                update_time   = current_timestamp(),
                flow_name     = '{sql_escape(FLOW_NAME)}'
            WHERE watermark_id = '{sql_escape(WATERMARK_ID)}'
            """)
        except Exception:
            pass

        # Persist metrics once on failure; in-memory metrics already include
        # all completed notebooks and the failing notebook.
        try:
            spark.sql(f"""
            UPDATE {CONTROL_RUN}
            SET
                status        = 'FAILED',
                ended_at      = current_timestamp(),
                duration      = '{sql_escape(duration_json)}',
                error_message = '{err}'
            WHERE run_id = '{sql_escape(run_id)}'
            """)
        except Exception:
            pass

    raise


StatementMeta(, d49eae0b-4fc8-4c38-96bb-cfbbcae9e9b0, 4, Finished, Available, Finished, False)

ExitValue: SUCCESS:93bcdad2-885e-45ef-854a-d09f34f931eb